# Environment

In [ ]:
import os
os.chdir("..")

In [ ]:
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification, pipeline
from datasets import Dataset

from src.utils.path import DATA_PATH, GO_EMOTIONS_PATH

# Load GoEmotions

In [ ]:
model = BertForSequenceClassification.from_pretrained(GO_EMOTIONS_PATH)
tokenizer = BertTokenizer.from_pretrained(GO_EMOTIONS_PATH)

In [ ]:
classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    truncation=True,
    max_length=256,
    top_k=None,
    return_all_scores=True,
    device=0
)

# Helper Functions

In [ ]:
def convert_emotion_scores_to_df(results_list):
    emotion_scores_data = []
    for single_text_results in results_list:
        scores_dict = {
            emotion["label"].lower(): emotion["score"]
            for emotion in single_text_results
        }
        emotion_scores_data.append(scores_dict)

    return pd.DataFrame.from_records(emotion_scores_data)

# Feature Engineering in Corpora

## Hate-BR

In [ ]:
hate = pd.read_csv(f"{DATA_PATH}/hate-br/processed/hate-br.csv")
hate_dataset = Dataset.from_pandas(hate[["text"]])

In [ ]:
emotions_hate = classifier(list(hate_dataset["text"]))
emotions_hate = convert_emotion_scores_to_df(emotions_hate)

In [ ]:
emotions_hate.to_csv(f"{DATA_PATH}/hate-br/features/ge.csv", index=False)

## Ited-BR

In [ ]:
ited = pd.read_csv(f"{DATA_PATH}/ited-br/processed/ited-br.csv")
dataset = Dataset.from_pandas(ited[["text"]])

In [ ]:
emotions_ited = classifier(list(dataset["text"]))
emotions_ited = convert_emotion_scores_to_df(emotions_ited)

In [ ]:
emotions_ited.to_csv(f"{DATA_PATH}/ited-br/features/ge.csv", index=False)

## Unified-Hate

In [ ]:
unified = pd.read_csv(f"{DATA_PATH}/unified-hate/processed/unified-hate.csv")
dataset = Dataset.from_pandas(unified[["text"]])

In [ ]:
emotions_unified = classifier(list(dataset["text"]))
emotions_unified = convert_emotion_scores_to_df(emotions_unified)

In [ ]:
emotions_unified.to_csv(f"{DATA_PATH}/unified-hate/features/ge.csv", index=False)